# Chandra OCR 2 sur GPU Kaggle

Le but est d'obtenir pour Chandra **exactement les memes chiffres** que pour
HunyuanOCR, PaddleOCR-VL et PP-StructureV3 : meme jeu d'images, meme code,
memes evaluations. Ce notebook ne fait donc que la **Phase 1**, l'inference
OCR, la seule qui ait besoin d'un GPU. Les evaluations tournent en quelques
secondes sur ton PC.

Chandra fait 5 milliards de parametres : il ne tient pas dans les 4 Go de la
RTX 3050 Ti, et le repli CPU tourne a ~15 min par image. C'est la raison
d'etre de ce notebook -- sur un T4, la meme page passe a une ou deux minutes.

## Les branches a produire

| run | jeu d'images | pretraitement |
|---|---|---|
| `multipage_chandra` | 17 pages deformees, 8 factures | non |
| `multipage_preproc_chandra` | idem | oui |
| `variants_chandra` | 25 pages a plat, 12 factures | non |
| `variants_warped_chandra` | 25 pages deformees | non |
| `variants_warped_preprocessed_chandra` | idem | oui |
| `variants_encombre_chandra` | 20 pages, 3 a 6 feuilles parasites | non |
| `variants_encombre_preproc_chandra` | idem | oui |
| `degradation_<lot>_chandra` | 24 lots x 8 pages, qualite degradee | non |

Les branches brut + pretraite vont par paire : `src/adaptive.py` choisit
entre les deux, il lui faut les deux.

> **Ne relancer que l'etude de degradation.** `SEULEMENT_DEGRADATION = True`
> dans la cellule des branches saute les sept branches classiques, deja
> mesurees : ~4 h 20 au lieu de ~8 h. Il suffit alors d'attacher le seul
> dataset `degradation`, la detection le prevoit.

## Ce que le jeu ENCOMBRE vient chercher

Mesure en local sur ces 20 images : la localisation de feuille **s'effondre**.
IoU 0,84 -> 0,37, et **aucune page sur vingt** n'a de recadrage exploitable,
contre 22 sur 25 en fond propre. Le plafond tombe aussi (0,86 -> 0,51), donc
ce n'est pas le selecteur mais la segmentation elle-meme.

La question posee au modele est donc precise, et elle n'a jamais ete posee :
**que devient la lecture quand on lui donne un recadrage FAUX ?** Pas absent
-- faux. C'est le cas le plus dangereux, parce que rien en aval ne le
signale.

Trois issues possibles, toutes instructives.

La lecture **brute** tient et la **pretraitee** s'effondre : le pretraitement
devient nuisible des qu'il y a du desordre, et la politique adaptative doit
savoir l'ecarter.

Les deux tiennent : Chandra lit la bonne facture meme mal recadree, et le
mauvais recadrage n'est qu'un gachis de calcul.

La pretraitee produit un tableau **coherent mais faux** -- celui d'une
feuille parasite. C'est le pire cas, et celui qu'il faut chercher : la
confiance arithmetique ne le verrait pas, puisque la facture parasite boucle
tout aussi bien.

## Avant de lancer

1. **Settings -> Accelerator -> GPU T4 x2** (P100 marche aussi, c'est plus lent)
2. **Settings -> Internet -> On** -- sans lui, ni `pip install`, ni `git clone`,
   ni le telechargement des 10 Go de poids depuis Hugging Face
3. **Add Input -> Datasets** : les jeux d'images voulus, zippes dans
   `kaggle/` du depot -- `images_multipage_warped.zip`,
   `images_variants_flat.zip`, `images_variants_warped.zip`, et
   `images_variants_encombre.zip` pour les fonds encombres,
   `images_degradation.zip` pour l'etude de robustesse
4. Depot GitHub a jour (`git push` depuis ton PC)

Puis **Save Version -> Save & Run All (Commit)** : le notebook tourne en
arriere-plan et tu peux fermer l'onglet. Sinon la session meurt avec la page.

## Licence -- a lire avant de publier des chiffres

Le **code** de Chandra est en Apache 2.0. Les **poids** sont sous une
OpenRAIL-M modifiee : libre pour la recherche, l'usage personnel et les
structures sous 2 M$ de revenus ou de levee, avec une clause de
non-concurrence vis-a-vis de l'API de Datalab.

Un stage de recherche academique entre dans ce cadre, et **aucune restriction
geographique** ne s'y applique -- contrairement a la licence Tencent de
HunyuanOCR, qui exclut l'UE, le Royaume-Uni et la Coree du Sud et rend donc
Kaggle risque puisqu'on n'y choisit pas la region. C'est un point a signaler
a ton encadrante : sur ce critere, Chandra est aussi confortable que
PaddleOCR-VL.

## 1. Verifier le GPU

In [ ]:
!nvidia-smi
import sys
print(sys.version)

## 2. Recuperer le code

Le depot ne contient que le code : les images vivent dans les datasets
Kaggle, et `results/` est ignore par git.

**Si le depot est prive**, ajoute un jeton GitHub dans *Add-ons -> Secrets*
sous le nom `GITHUB_TOKEN`. La cellule le detecte toute seule.

`GIT_TERMINAL_PROMPT=0` est indispensable : sans lui, un depot prive fait
attendre `git clone` sur une invite de mot de passe que personne ne lira, et
le notebook reste bloque sans le moindre message d'erreur.

In [ ]:
import os, subprocess

# Jamais d'invite interactive : on veut une erreur, pas un blocage.
os.environ["GIT_TERMINAL_PROMPT"] = "0"

OWNER, REPO, SECRET = "Halfaa", "PTITOcr", "GITHUB_TOKEN"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(SECRET)
    print(f"Secret '{SECRET}' trouve ({len(token)} caracteres)")
except ImportError:
    print("kaggle_secrets indisponible : tu n'es pas sur Kaggle ?")
except Exception as e:
    print(f"Secret '{SECRET}' illisible -> {type(e).__name__}: {e}")
    print("\nA verifier, dans cet ordre :")
    print("  1. Add-ons -> Secrets : le secret est-il RATTACHE a CE notebook")
    print("     (interrupteur a droite de son nom) ? C'est l'oubli habituel.")
    print(f"  2. Le label est-il exactement '{SECRET}' ?")
    print("  3. Cree APRES le demarrage de la session ? Run -> Restart Session.")

url = (f"https://{token}@github.com/{OWNER}/{REPO}.git" if token
       else f"https://github.com/{OWNER}/{REPO}.git")

DEPOT = f"/kaggle/working/{REPO}"
%cd /kaggle/working
!rm -rf {REPO}
r = subprocess.run(["git", "clone", "--depth", "1", url, REPO],
                   capture_output=True, text=True, timeout=180)
if r.returncode != 0:
    # On masque le jeton avant tout affichage : la sortie d'un notebook
    # Kaggle est conservee dans la version publiee.
    msg = r.stderr.replace(token, "***") if token else r.stderr
    raise SystemExit("Clone impossible.\n" + msg)

%cd {DEPOT}
!ls

## 3. Installer Chandra

`chandra-ocr[hf]` tire un `transformers` recent : Chandra 2 derive de
Qwen3.5, dont l'architecture n'existe pas dans les `transformers` 4.x.

**C'est pour cette raison que ce notebook est separe de celui de
PaddleOCR.** Paddle exige `numpy < 2` et un `transformers` 4.x ; les deux
piles ne cohabitent pas, exactement comme les deux venv du projet en local.

Le risque a surveiller ici est different de celui de Paddle : l'installation
peut **remplacer le torch CUDA preinstalle par Kaggle** par une roue CPU.
Rien ne le signale -- le modele se charge, tourne sur le processeur, et on
decouvre au bout d'une heure que le GPU n'a jamais servi. La cellule
suivante verifie ce point avant qu'on engage le moindre calcul.

In [ ]:
import subprocess, sys, importlib.metadata as md

def pip(*args):
    """pip qui ECHOUE bruyamment.

    `!pip install` n'interrompt pas la cellule quand il rate : le run
    continue et ne s'effondre que bien plus tard, sur un ModuleNotFoundError
    qui ne dit rien de la cause.
    """
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join(r.stdout.splitlines()[-5:]))
        print("\n".join(r.stderr.splitlines()[-10:]))
    return r.returncode == 0

torch_avant = md.version("torch")
print(f"torch preinstalle par Kaggle : {torch_avant}")

assert pip("-q", "chandra-ocr[hf]"), "installation de chandra-ocr echouee"

torch_apres = md.version("torch")
if torch_apres != torch_avant:
    print(f"\nATTENTION : torch est passe de {torch_avant} a {torch_apres}.")
    print("Si la verification suivante annonce cuda: False, reinstalle la")
    print("roue CUDA d'origine :")
    print(f"    !pip install -q --force-reinstall torch=={torch_avant}")

for pkg in ("chandra-ocr", "transformers", "torch", "accelerate", "pillow"):
    try:
        print(f"{pkg:<16}{md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<16}absent")

In [ ]:
# Verification AVANT de lancer des heures de calcul.
#
# Un torch sans CUDA ne leve aucune erreur : il fait tourner Chandra sur le
# processeur, a ~15 min la page au lieu d'une ou deux. Sur 109 pages, c'est
# la difference entre trois heures et trois semaines. On refuse de demarrer.
import subprocess, sys

check = subprocess.run([sys.executable, "-c", """
import torch, transformers
print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
print('transformers', transformers.__version__)
assert torch.cuda.is_available(), 'torch ne voit pas le GPU'
n = torch.cuda.device_count()
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.0f} Go, '
          f'capability {p.major}.{p.minor}')
p = torch.cuda.get_device_properties(0)
assert p.total_memory > 13e9, f'{p.total_memory/1e9:.0f} Go : trop peu pour 5B'
print()
print('dtype retenu :', 'bfloat16' if p.major >= 8 else
      'float16 (pas de bf16 materiel avant la capability 8.0)')
"""], capture_output=True, text=True)

print(check.stdout)
if check.returncode != 0:
    print(check.stderr[-1500:])
    raise SystemExit(
        "Environnement inutilisable -- ne lance pas le benchmark.\n"
        "  cuda: False        -> l'install a remplace torch, voir la cellule\n"
        "                        precedente\n"
        "  aucun GPU          -> Settings -> Accelerator -> GPU T4 x2\n"
        "  moins de 13 Go     -> demande un T4 ou un P100, pas un GPU partage")
print("Environnement OK.")

## 4. Reperer les jeux d'images

Kaggle monte les datasets attaches sous `/kaggle/input/`, **sous le nom que
tu leur as donne en les televersant** -- pas sous le nom du zip. C'est le
piege : `images_variants_flat.zip` peut arriver en `variant-flat`, et
`images_variants_warped.zip` en `wrapeddataset`.

`MOTIFS` fait donc le pont. Chaque jeu accepte plusieurs ecritures ; si ton
dataset porte encore un autre nom, ajoute-le a la liste correspondante
plutot que de renommer le dataset.

Deux garde-fous, pour la meme raison : un run de plusieurs heures sur les
mauvaises images ne se rattrape pas, alors qu'une erreur immediate se corrige
en dix secondes.

**Le nombre de pages doit correspondre.** Tu as sans doute encore attaches
`no-illum` et `image-preprocess-test`, qui font 17 pages comme le jeu
multi-pages mais ne sont *pas* lui : l'un est rendu sans eclairage, l'autre
deja pretraite. Compter les pages ne suffit pas a les distinguer -- seul le
nom le peut. La cellule affiche donc explicitement les dossiers qu'elle
ignore.

**Un jeu absent ne bloque pas**, ses branches sont sautees. Mais si aucun
n'est trouve, on s'arrete ici : c'est ce qui a fait planter la version
precedente, trois cellules plus loin et sans dire pourquoi.

In [ ]:
from pathlib import Path

MOTIFS = {
    # jeu               -> ecritures acceptees dans le chemin du dataset
    "multipage":       ["multipage_warped", "multipage-warped", "multipage"],
    "variants_flat":   ["variants_flat", "variant-flat", "variant_flat"],
    # « encombre » AVANT « warped » : le jeu encombre est lui aussi deforme,
    # donc son nom pourrait contenir les deux. L'ordre du dictionnaire tranche.
    "variants_encombre": ["variants_encombre", "variant-encombre", "encombre",
                          "cluttered"],
    "variants_warped": ["variants_warped", "variant-warped", "wrapeddataset",
                        "warped"],
}

# Plusieurs comptes acceptes pour un meme jeu quand un rendu partiel est
# legitime. Le PREMIER de la liste est le jeu complet.
PAGES_ATTENDUES = {"multipage": [17], "variants_flat": [25],
                   "variants_warped": [25], "variants_encombre": [25, 20]}

# Un jeu partiel n'est pas une erreur, mais il n'est pas neutre non plus : une
# facture amputee d'une page est notee quand meme, sur ce qui reste, et sort
# un chiffre bas qui ressemble a une faiblesse du modele.
PARTIELS = {
    ("variants_encombre", 20):
        "Il manque les pages 5 a 9 -- le rendu Blender trie les textures par\n"
        "  NOM, qui met page_5..page_9 en fin de liste, et -Nombre 20 les a\n"
        "  coupees. Les factures 3 et 4 sont donc absentes, sans consequence,\n"
        "  mais la facture 5 est AMPUTEE de sa premiere page : son score sera\n"
        "  bas pour cette raison, pas a cause du modele.\n"
        "  Pour completer, sur ton PC, dans doc3D-renderer :\n"
        "    .\\render_variants.ps1 -FolderId variants_encombre -Parasites -1 "
        "-Depuis 20 -Nombre 5",
}

dossiers_avec_images = {}
for png in Path("/kaggle/input").rglob("page_*.png"):
    dossiers_avec_images[png.parent] = dossiers_avec_images.get(png.parent, 0) + 1

# Ce garde-fou vient AVANT le tri des lots de degradation, et pas apres.
# Mis apres, il se declenchait a tort quand SEUL le dataset degradation etait
# attache : ses 24 sous-dossiers venaient d'etre retires du dictionnaire, qui
# se retrouvait vide, et le notebook annoncait « aucune image » avec 192
# images sous la main.
assert dossiers_avec_images, "Aucune image : as-tu ajoute les datasets via Add Input ?"

# L'etude de degradation est a part : UN dataset, 24 SOUS-dossiers de 8
# pages (niveau0, flou1..5, bruit1..5, exposition1..5, resolution1..5,
# jpeg1..3). On les met de cote avant les garde-fous ci-dessous, sinon
# chaque sous-dossier compterait comme un jeu a lui seul et ferait echouer
# la detection.
LOTS_DEGRADATION = {}
for dossier in list(dossiers_avec_images):
    if "degradation" in str(dossier).lower().replace("-", "_"):
        LOTS_DEGRADATION[dossier.name] = dossier
        del dossiers_avec_images[dossier]

if LOTS_DEGRADATION:
    tailles = {n: len(list(d.glob("page_*.png")))
               for n, d in LOTS_DEGRADATION.items()}
    print(f"\nEtude de degradation : {len(LOTS_DEGRADATION)} lots")
    assert "niveau0" in LOTS_DEGRADATION, (
        "Le lot 'niveau0' manque. C'est la REFERENCE : sans lui la courbe "
        "se compare a un\nchiffre mesure ailleurs, et le moindre ecart de "
        "pipeline se lirait comme un\neffet de la degradation.")
    if len(set(tailles.values())) > 1:
        raise SystemExit(
            "Les lots n'ont pas tous le meme nombre de pages :\n"
            + "\n".join(f"  {n:<14}{k}" for n, k in sorted(tailles.items()))
            + "\n\nLa comparaison doit etre APPARIEE -- memes pages a tous "
              "les paliers.\nUn lot perime d'une grille precedente traine "
              "sans doute dans le zip.")
    print(f"  {len(LOTS_DEGRADATION)} lots x {next(iter(tailles.values()))} pages")

# Apres le tri : les 24 lots de degradation ne polluent pas cette liste,
# ils ont eu leur propre ligne de resume juste au-dessus.
print("\nDossiers trouves sous /kaggle/input :")
for dossier, n in sorted(dossiers_avec_images.items(), key=lambda kv: -kv[1]):
    print(f"  {n:>4} images   {dossier}")
if not dossiers_avec_images:
    print("  (aucun -- seule l'etude de degradation est attachee)")

def normalise(chemin):
    return str(chemin).lower().replace("-", "_")

JEUX, manquants, pris = {}, [], set()
for nom, ecritures in MOTIFS.items():
    # Les ecritures sont essayees dans l'ordre : la plus specifique d'abord.
    # "warped" tout seul est en dernier parce qu'il attraperait aussi
    # "multipage_warped" -- l'ordre est ce qui evite l'ambiguite.
    retenus = []
    for ecriture in ecritures:
        retenus = [d for d in dossiers_avec_images
                   if ecriture.replace("-", "_") in normalise(d)
                   and d not in pris]
        if retenus:
            break
    if not retenus:
        manquants.append(nom)
        continue
    if len(retenus) > 1:
        raise SystemExit(
            f"'{nom}' correspond a {len(retenus)} dossiers :\n"
            + "\n".join(f"  {d}" for d in retenus)
            + "\n\nDetache les datasets en trop, ou precise MOTIFS.")
    JEUX[nom] = retenus[0]
    pris.add(retenus[0])

print("\nJeux retenus :")
for nom, chemin in JEUX.items():
    n = len(list(chemin.glob("page_*.png")))
    attendus = PAGES_ATTENDUES[nom]
    marque = "" if n in attendus else f"   <-- {attendus[0]} attendues !"
    print(f"  {nom:<18}{n:>4} pages   {chemin.name}{marque}")
    if n not in attendus:
        raise SystemExit(
            f"'{nom}' a {n} pages au lieu de {attendus[0]}. C'est probablement "
            "le mauvais dataset :\nun jeu qui n'est pas celui des autres "
            "modeles produit des chiffres non comparables,\nce qui est pire "
            "qu'une absence de chiffres.")
    if n != attendus[0]:
        print(f"  ATTENTION : jeu partiel, {n} pages sur {attendus[0]}.")
        print("  " + PARTIELS.get((nom, n), "Cause inconnue."))

ignores = [d for d in dossiers_avec_images if d not in pris]
if ignores:
    print("\nDossiers IGNORES (aucun motif ne colle) :")
    for d in ignores:
        print(f"  {dossiers_avec_images[d]:>4} images   {d}")

if manquants:
    print("\nJeux ABSENTS, leurs branches seront sautees :", ", ".join(manquants))
    print("  Pour les ajouter : Add Input -> Datasets, en televersant le zip")
    print("  correspondant depuis kaggle/ du depot :")
    for nom in manquants:
        print(f"    {nom:<18}kaggle/images_{nom.replace('multipage', 'multipage_warped')}.zip")

assert JEUX or LOTS_DEGRADATION, (
    "Aucun jeu reconnu. Les dossiers listes plus haut ne correspondent a "
    "aucune\necriture de MOTIFS : ajoute le nom reel de tes datasets dans "
    "les listes,\nen haut de cette cellule.")

if not JEUX:
    print("\nSeule l'etude de degradation est attachee. C'est un cas "
          "PREVU :\nles branches classiques seront sautees, rien ne sera "
          "recalcule pour rien.")

## 5. Pretraiter sur place

Le depot est deja clone, donc `src/preprocess.py` est disponible ici : on
pretraite dans la session au lieu de televerser un second dataset. Deux
raisons.

Le code de pretraitement est **toujours celui du depot**. Des images
pretraitees televersees se perimeraient a la premiere modification du module,
sans que rien ne le signale -- on mesurerait alors une version du
pretraitement qui n'existe plus.

Et les deux lectures sortent de la meme session, ce dont `src/adaptive.py` a
besoin : la politique choisit entre lecture brute et lecture pretraitee, donc
il lui faut les deux, produites dans les memes conditions.

> **Comparabilite.** Les chiffres pretraites publies pour HunyuanOCR et
> PaddleOCR-VL datent d'avant les derniers reglages du module (IoU 0,79 puis
> 0,90). Pour comparer les modeles sur cette branche, il faudra soit rejouer
> leur pretraitement avec le code actuel, soit ne comparer que la branche
> brute. La branche brute, elle, est comparable telle quelle.

In [ ]:
import subprocess, sys, time
from pathlib import Path

A_PRETRAITER = ["multipage", "variants_warped", "variants_encombre"]

# L'etude de degradation ne s'en sert pas : elle se mesure en lecture
# brute. Si aucun jeu classique n'est attache, cette cellule ne fait rien,
# ce qui est le comportement voulu et pas un oubli.

PRETRAITES = {}
for nom in A_PRETRAITER:
    if nom not in JEUX:
        continue
    source = JEUX[nom]
    sortie = Path(f"/kaggle/working/preproc_{nom}")
    debut = time.time()
    r = subprocess.run(
        [sys.executable, "src/preprocess.py",
         "--images-dir", str(source), "--out", str(sortie), "--debug"],
        cwd=DEPOT, capture_output=True, text=True)
    print(f"--- {nom}")
    print("\n".join(r.stdout.splitlines()[-6:]))
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit(f"Le pretraitement de {nom} a echoue.")

    n = len(list(sortie.glob("page_*.png")))
    attendu = len(list(source.glob("page_*.png")))
    assert n == attendu, f"{n} images pretraitees pour {attendu} attendues"
    PRETRAITES[nom] = sortie
    print(f"    {n} pages en {time.time() - debut:.0f}s -> {sortie}\n")

## 6. Etalonner avant de s'engager

Une page de Chandra sur T4, c'est une ou deux minutes -- mais l'estimation
depend du GPU tire au sort, de la taille des images et de la longueur du
tableau. Sur 109 pages, se tromper d'un facteur deux fait la difference entre
tenir dans la session et se faire couper au milieu.

Cette cellule lit **une seule page**, chronometre, et projette le total. Le
chargement du modele (10 Go a telecharger puis a poser sur le GPU) est
compte a part : il ne se paie qu'une fois.

> **Le message « The fast path is not available »** au chargement n'est pas
> une erreur. Chandra 2 derive de Qwen3.5, un modele a attention lineaire
> hybride, dont les noyaux optimises (`flash-linear-attention`,
> `causal-conv1d`) ne sont pas installes ici : transformers retombe sur son
> implementation PyTorch, correcte mais plus lente. Les installer suppose
> une compilation CUDA incertaine sur un T4 (capability 7.5), et surtout
> **cela changerait l'implementation entre deux runs** -- donc entre deux
> modeles compares. On mesure avec le chemin PyTorch, et la projection
> ci-dessous dit si c'est tenable. Si ca ne l'est pas, la bonne reponse est
> de reduire le nombre de branches, pas de changer de noyau en cours de
> protocole.

In [ ]:
import sys, time
sys.path.insert(0, f"{DEPOT}/scripts")
sys.path.insert(0, f"{DEPOT}/src")

from pathlib import Path
from PIL import Image
import bench_deformed as bench

# On choisit la page d'essai AVANT de charger 10 Go de poids : si le jeu
# n'est pas la, autant l'apprendre en une seconde. La version precedente
# faisait l'inverse et mourait sur un StopIteration nu, apres deux minutes
# de chargement et sans nommer la cause.
jeu_essai = next((n for n in ("multipage", "variants_warped", "variants_flat")
                  if n in JEUX), None)
if jeu_essai:
    dossier_essai = JEUX[jeu_essai]
else:
    # Run degradation seul : on etalonne sur la page de REFERENCE, qui
    # n'est degradee par rien. Etalonner sur une page abimee donnerait un
    # temps par page fausse -- une image bruitee produit plus de jetons.
    jeu_essai = "niveau0"
    dossier_essai = LOTS_DEGRADATION.get("niveau0")
assert dossier_essai, "Aucun jeu disponible : voir la cellule de detection."
pages_essai = sorted(dossier_essai.glob("page_*.png"))
assert pages_essai, f"{dossier_essai} ne contient aucune page_*.png"
premiere = pages_essai[0]

t0 = time.time()
modele = bench._load_chandra()
t_chargement = time.time() - t0
print(f"\nChargement : {t_chargement:.0f}s")

print(f"Page d'essai : {premiere.name} ({Image.open(premiere).size}) "
      f"[jeu {jeu_essai}]")

t0 = time.time()
texte = bench._chandra_predict(modele, premiere)
par_page = time.time() - t0
print(f"Lecture : {par_page:.0f}s")
print(f"\n--- 400 premiers caracteres ---\n{texte[:400]}")

# Un tableau ou rien : si la sortie ne contient ni <table> ni pipes markdown,
# `table_merge` n'aura rien a recoller et tout le reste de la chaine mesurera
# du vide. Mieux vaut le savoir maintenant qu'apres trois heures.
a_du_tableau = "<table" in texte.lower() or texte.count("|") > 10
print(f"\nTableau detecte dans la sortie : {'oui' if a_du_tableau else 'NON'}")
if not a_du_tableau:
    print("  -> verifie prompt_type dans _chandra_predict avant de continuer.")

In [ ]:
# Projection, branche par branche.
BRANCHES = [
    # (nom du run,                            jeu,               pretraite)
    ("multipage_chandra",                     "multipage",       False),
    ("multipage_preproc_chandra",             "multipage",       True),
    ("variants_chandra",                      "variants_flat",   False),
    ("variants_warped_chandra",               "variants_warped", False),
    ("variants_warped_preprocessed_chandra",  "variants_warped", True),
    # Le jeu a fonds encombres. La branche PRETRAITEE est celle qui compte :
    # mesure en local, la localisation de feuille s'y effondre (IoU 0,84 ->
    # 0,37, aucune page exploitable). La question posee au modele est donc
    # « que devient la lecture quand on lui donne un recadrage FAUX » --
    # pas absent, faux, ce qui est le cas le plus dangereux.
    ("variants_encombre_chandra",             "variants_encombre", False),
    ("variants_encombre_preproc_chandra",     "variants_encombre", True),
]

# L'etude de degradation, en LECTURE BRUTE uniquement.
#
# Pas de branche pretraitee, et le choix est budgetaire, pas negligent :
# on mesure ici la robustesse de la LECTURE a la qualite d'image. Doubler
# pour le pretraitement doublerait le temps GPU afin de repondre a une
# AUTRE question -- la correction d'eclairage aggrave-t-elle le bruit --
# qui merite son propre run une fois connus les points de rupture.
BRANCHES_DEGRADATION = [(f"degradation_{nom}_chandra", nom)
                        for nom in sorted(LOTS_DEGRADATION)]

# Ne faire QUE l'etude de degradation.
#
# Les sept branches classiques sont deja mesurees et rapatriees : les
# rejouer coute ~3 h de GPU pour reproduire des chiffres identiques. Mettre
# False pour tout refaire, par exemple apres un changement du code de
# lecture qui invaliderait les runs precedents.
SEULEMENT_DEGRADATION = True

total_pages = 0
print(f"{'run':<40}{'pages':>7}{'estime':>10}")
print("-" * 57)
for run, jeu, pretraite in BRANCHES:
    if SEULEMENT_DEGRADATION:
        print(f"{run:<40}{'--':>7}{'desactive':>10}")
        continue
    if jeu not in JEUX or (pretraite and jeu not in PRETRAITES):
        print(f"{run:<40}{'--':>7}{'jeu absent':>10}")
        continue
    n = len(list(JEUX[jeu].glob("page_*.png")))
    total_pages += n
    print(f"{run:<40}{n:>7}{n * par_page / 60:>9.0f}m")

if BRANCHES_DEGRADATION:
    n_deg = sum(len(list(LOTS_DEGRADATION[j].glob("page_*.png")))
                for _, j in BRANCHES_DEGRADATION)
    total_pages += n_deg
    print(f"{f'degradation ({len(BRANCHES_DEGRADATION)} lots)':<40}"
          f"{n_deg:>7}{n_deg * par_page / 60:>9.0f}m")
print("-" * 57)
print(f"{'TOTAL':<40}{total_pages:>7}{total_pages * par_page / 3600:>9.1f}h")
print()
print("Une session Kaggle avec GPU est coupee a 12h, et le quota")
print("hebdomadaire est de 30h. Si le total depasse ~9h, coupe BRANCHES en")
print("deux notebooks : les runs sont independants et s'evaluent separement.")

## 7. Lancer

`scripts/bench_deformed.py` reecrit son rapport apres chaque image : une
interruption ne coute qu'une page, et **relancer reutilise ce qui est deja
fait**. C'est ce qui permet de reprendre un run coupe.

Pour reprendre entre deux sessions, ajoute la sortie de la version
precedente comme dataset d'entree (*Add Input -> Your Work -> Notebook
Output*) et decompresse-la dans `results/` avant cette cellule : le script
sautera les pages deja lues.

`BRANCHES_A_FAIRE = None` fait tout. Mets une liste de noms de runs pour
n'en faire qu'une partie.

> **Lire l'avancement.** Le script imprime `[i/N] page.png` puis
> `-> done in Xs`. Ces lignes viennent d'un processus fils, lance avec `-u`
> pour qu'elles sortent au fil de l'eau : sans ce drapeau, Python les
> bufferise et ne les livre qu'a la mort du processus, toutes horodatees a
> la meme seconde. Un silence de vingt minutes ressemble alors a un
> blocage. Le modele est aussi **recharge a chaque branche** -- une a deux
> minutes de silence entre deux branches est normale.

In [ ]:
BRANCHES_A_FAIRE = None    # ex. ["multipage_chandra", "multipage_preproc_chandra"]

import subprocess, sys, time

# Le modele charge dans la cellule d'etalonnage occupe le GPU ; le script
# fils en chargerait un second et tomberait en OOM. On libere d'abord.
try:
    del modele
except NameError:
    pass
import gc, torch
gc.collect()
torch.cuda.empty_cache()
libre = torch.cuda.mem_get_info()[0] / 1e9
print(f"VRAM libre : {libre:.1f} Go\n")
assert libre > 12, (
    f"Seulement {libre:.1f} Go libres : le modele de la cellule "
    "d'etalonnage n'a pas ete rendu.\nRun -> Restart Session, puis rejoue "
    "tout SAUF la cellule d'etalonnage.")

runs_produits = []
if SEULEMENT_DEGRADATION:
    print(f"SEULEMENT_DEGRADATION : les {len(BRANCHES)} branches "
          "classiques sont sautees.\n")
for run, jeu, pretraite in BRANCHES:
    if SEULEMENT_DEGRADATION:
        continue
    if BRANCHES_A_FAIRE and run not in BRANCHES_A_FAIRE:
        continue
    source = PRETRAITES.get(jeu) if pretraite else JEUX.get(jeu)
    if source is None:
        print(f"[saute] {run} : jeu '{jeu}' indisponible")
        continue

    print(f"\n{'=' * 64}\n{run}\n  images : {source}\n{'=' * 64}", flush=True)
    debut = time.time()
    # -u : sortie NON tamponnee du processus fils.
    #
    # Sans lui, Python bufferise stdout par blocs des qu'il n'ecrit pas dans
    # un terminal -- ce qui est le cas ici. Les `[i/N]` de progression ne
    # sortent alors qu'a la mort du processus, tous d'un coup et tous
    # horodates a la meme seconde. On croit le run bloque pendant vingt
    # minutes alors qu'il avance normalement, et on n'a aucun moyen de
    # distinguer ce silence-la d'un vrai blocage.
    subprocess.run(
        [sys.executable, "-u", "scripts/bench_deformed.py", "--model", "chandra",
         "--images-dir", str(source), "--output", run],
        cwd=DEPOT, check=True)
    print(f"\n{run} termine en {(time.time() - debut) / 60:.0f} min", flush=True)
    runs_produits.append(run)

for run, lot in BRANCHES_DEGRADATION:
    if BRANCHES_A_FAIRE and run not in BRANCHES_A_FAIRE:
        continue
    source = LOTS_DEGRADATION[lot]
    print(f"\n{'=' * 64}\n{run}\n  images : {source}\n{'=' * 64}", flush=True)
    debut = time.time()
    subprocess.run(
        [sys.executable, "-u", "scripts/bench_deformed.py", "--model", "chandra",
         "--images-dir", str(source), "--output", run],
        cwd=DEPOT, check=True)
    print(f"\n{run} termine en {(time.time() - debut) / 60:.0f} min", flush=True)
    runs_produits.append(run)

print("\nRuns produits :", runs_produits)

## 8. Rapatrier

Tout ce qui est dans `/kaggle/working` se telecharge depuis l'onglet
**Output** a la fin du run. Un zip par run : on peut ainsi n'en rapatrier
qu'un si un seul a change.

In [ ]:
import shutil
from pathlib import Path

base = Path(DEPOT) / "results"
for run in runs_produits:
    src = base / run
    assert src.exists(), f"{src} absent : le benchmark a-t-il tourne ?"
    n = len([d for d in (src / "pages").iterdir() if d.is_dir()])
    out = shutil.make_archive(f"/kaggle/working/{run}", "zip", base, run)
    print(f"{Path(out).name:<48}{n:>4} pages{Path(out).stat().st_size / 1e6:>8.1f} Mo")

# Le depot clone et les images pretraitees pesent pour rien dans l'Output.
shutil.rmtree(DEPOT, ignore_errors=True)
for d in Path("/kaggle/working").glob("preproc_*"):
    shutil.rmtree(d, ignore_errors=True)

print("\nContenu de /kaggle/working :")
for f in sorted(Path("/kaggle/working").iterdir()):
    print(" ", f.name)

## 9. Ensuite, sur ton PC

Decompresse les zips dans `MedicalRecord/results/`, puis :

```powershell
.\venv\Scripts\activate
python scripts\eval_all.py --model chandra
```

`eval_all.py` enchaine TEDS, fusion multi-pages, champs d'en-tete,
validation arithmetique et politique adaptative, avec la bonne verite
terrain pour chaque jeu. C'est le meme protocole pour tous les modeles :
c'est ce qui rend les chiffres comparables.

L'etude de degradation a son propre depouillement, qui trace la courbe :

```powershell
python scripts\eval_degradation.py --figure
```

Pour la comparaison cote a cote :

```powershell
python scripts\eval_all.py --model chandra --model hunyuan --model paddleocr
```